# Lire et enregistrer un fichier

In [ ]:
import sys
sys.path.append("..")

import os
import pandas as pd
from _data import load_accidents

df = load_accidents()
df.shape

## 1. CSV — lecture, écriture, pièges

CSV est le format universel d'échange lisible par un humain.
Il est aussi le format le plus lent, le plus lourd et le moins fiable sur les types.

In [ ]:
# Écriture
df.to_csv("accidents_export.csv", index=False)  # index=False : ne pas écrire l'index 0,1,2
print(f"Taille CSV : {os.path.getsize('accidents_export.csv') / 1024**2:.1f} Mo")

In [ ]:
# Lecture de base
df_relu = pd.read_csv("accidents_export.csv")
df_relu.shape

In [ ]:
# Piège 1 : les types sont inférés à la lecture — souvent mal
# Comparer les dtypes avant et après
pd.DataFrame({
    "original": df.dtypes,
    "relu_csv":  df_relu.dtypes,
})

In [ ]:
# Solution : spécifier les types à la lecture
df_typed = pd.read_csv(
    "accidents_export.csv",
    dtype={"dep": "string", "an": "Int16", "mois": "Int8"},
    usecols=["Num_Acc", "dep", "mois", "lum", "col"],  # charger seulement ce dont on a besoin
)
df_typed.dtypes

In [ ]:
# Piège 2 : les séparateurs décimaux varient selon la locale
# Un fichier français peut utiliser ',' comme décimale et ';' comme séparateur
df_fr = pd.DataFrame({"valeur": [1.5, 2.3], "label": ["a", "b"]})
df_fr.to_csv("test_fr.csv", sep=";", decimal=",", index=False)

# Lecture avec les bons paramètres
pd.read_csv("test_fr.csv", sep=";", decimal=",")

In [ ]:
# Piège 3 : les dates ne sont pas parsées automatiquement
# Il faut les spécifier ou les convertir après lecture
df_dates = pd.DataFrame({"date": ["2024-01-15", "2024-06-30"], "val": [1, 2]})
df_dates.to_csv("test_dates.csv", index=False)

# Sans parse_dates
df_sans = pd.read_csv("test_dates.csv")
print("Sans parse :", df_sans["date"].dtype)

# Avec parse_dates
df_avec = pd.read_csv("test_dates.csv", parse_dates=["date"])
print("Avec parse :", df_avec["date"].dtype)

## 2. Parquet — le format moderne par défaut

Parquet est un format **binaire columnar**. Il stocke les données colonne par colonne,
avec compression et types préservés. C'est le standard de facto de l'écosystème data moderne
(Spark, DuckDB, polars, BigQuery, Snowflake — tous parlent Parquet).

In [ ]:
# Écriture
df.to_parquet("accidents.parquet", index=False)
print(f"Taille Parquet : {os.path.getsize('accidents.parquet') / 1024**2:.1f} Mo")
print(f"Taille CSV     : {os.path.getsize('accidents_export.csv') / 1024**2:.1f} Mo")
print(f"Ratio          : {os.path.getsize('accidents_export.csv') / os.path.getsize('accidents.parquet'):.1f}x")

In [ ]:
# Les types sont préservés à la lecture — pas d'inférence
df_parquet = pd.read_parquet("accidents.parquet")
pd.DataFrame({
    "original":    df.dtypes,
    "relu_parquet": df_parquet.dtypes,
})

In [ ]:
%%timeit
pd.read_csv("accidents_export.csv")

In [ ]:
%%timeit
pd.read_parquet("accidents.parquet")

In [ ]:
%%timeit
# Parquet : sélection de colonnes GRATUITE — le reste n'est pas décompressé
pd.read_parquet("accidents.parquet", columns=["dep", "mois", "lum"])

### Moteurs Parquet : `pyarrow` vs `fastparquet`

In [ ]:
# pandas supporte deux moteurs — pyarrow est recommandé
df.to_parquet("accidents_arrow.parquet", engine="pyarrow")    # défaut
print("pyarrow OK")

try:
    df.to_parquet("accidents_fast.parquet", engine="fastparquet")
    print("fastparquet OK")
except ImportError:
    print("fastparquet non installé (moins courant)")

## 3. Excel — quand on ne peut pas l'éviter

In [ ]:
# Écriture Excel (nécessite openpyxl)
try:
    # On exporte un sous-ensemble — Excel est lent sur les gros volumes
    df.head(1000).to_excel("accidents_extrait.xlsx", sheet_name="Accidents", index=False)
    print(f"Excel écrit : {os.path.getsize('accidents_extrait.xlsx') / 1024:.0f} Ko")
except ImportError:
    print("openpyxl non installé : uv add openpyxl")

In [ ]:
# Lecture Excel
try:
    df_excel = pd.read_excel("accidents_extrait.xlsx", sheet_name="Accidents")
    df_excel.shape
except FileNotFoundError:
    print("Fichier Excel non trouvé — openpyxl requis pour l'étape précédente")

In [ ]:
# Lire plusieurs feuilles
try:
    with pd.ExcelWriter("multi_feuilles.xlsx", engine="openpyxl") as writer:
        df.query("mois <= 6").head(500).to_excel(writer, sheet_name="S1", index=False)
        df.query("mois > 6").head(500).to_excel(writer, sheet_name="S2", index=False)

    feuilles = pd.read_excel("multi_feuilles.xlsx", sheet_name=None)  # dict {nom: DataFrame}
    print({k: v.shape for k, v in feuilles.items()})
except ImportError:
    print("openpyxl requis")

## 4. Autres formats utiles

In [ ]:
# JSON
df.head(10).to_json("accidents_extrait.json", orient="records", force_ascii=False)
pd.read_json("accidents_extrait.json").shape

In [ ]:
# Feather — encore plus rapide que Parquet pour les échanges in-process
# (pas compressé, pas portable entre machines, idéal pour le cache local)
try:
    df.to_feather("accidents.feather")
    print(f"Feather : {os.path.getsize('accidents.feather') / 1024**2:.1f} Mo")

    %timeit pd.read_feather("accidents.feather")
except Exception as e:
    print(f"Feather non disponible : {e}")

## 5. Règle de choix du format

> **Par défaut : Parquet.**
> CSV uniquement pour échanger avec un humain ou un outil qui ne lit que ça.
> Excel uniquement quand le destinataire ne sait utiliser que ça.

| Situation | Format |
|---|---|
| Stocker des données pour réutilisation | **Parquet** |
| Échanges entre outils data (DuckDB, Spark, polars) | **Parquet** |
| Envoyer à un collègue non-technique | **Excel** (ou CSV si petit) |
| Intégration avec une API web, config | **JSON** |
| Cache local temporaire (perf maximale) | **Feather** |
| Source externe sans autre choix | **CSV** (avec `dtype=` et `usecols=`) |

In [ ]:
# Nettoyage
for f in ["accidents_export.csv", "accidents.parquet", "accidents_arrow.parquet",
          "accidents_extrait.xlsx", "multi_feuilles.xlsx", "accidents_extrait.json",
          "accidents.feather", "test_fr.csv", "test_dates.csv"]:
    if os.path.exists(f):
        os.remove(f)

## Quiz

<details><summary>Vous recevez un CSV d'un client avec des virgules comme séparateur décimal et des points-virgules comme séparateur de colonnes. Comment le lire correctement ?</summary>

```python
pd.read_csv("fichier.csv", sep=";", decimal=",")
```

C'est un format typiquement exporté depuis Excel français. Sans ces paramètres,
toutes les colonnes numériques seront lues comme des strings.

</details>

<details><summary>Vous avez un fichier Parquet de 2 Go avec 50 colonnes. Vous n'en utilisez que 3. Comment lire seulement ces 3 colonnes ?</summary>

```python
pd.read_parquet("fichier.parquet", columns=["col1", "col2", "col3"])
```

En Parquet, les colonnes non demandées ne sont **pas décompressées** — la lecture est proportionnelle
aux colonnes choisies, pas à la taille totale du fichier. C'est impossible avec CSV.

</details>